# Universal Adversarial Cloak Demo
This notebook validates universal adversarial cloaking on **all local fixture images** across face and general modalities.

## Setup
Load dependencies, fixture paths, and helper utilities used in the walkthrough.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from uacloak.cloaking import CloakHyperparameters, cloak_face_tensor, cloak_general_image
from uacloak.evaluation import compute_ssim_score
from uacloak.models import get_clip_model
from uacloak.pipeline import (
    classify_image_type,
    cosine_similarity,
    detect_primary_face,
    extract_clip_embedding_numpy,
    extract_embedding_numpy,
)

ROOT = Path.cwd()
FACE_DIR = ROOT / 'tests' / 'fixtures' / 'faces'
GENERAL_DIR = ROOT / 'tests' / 'fixtures' / 'general'

FACE_IMAGES = sorted(FACE_DIR.glob('*.jpg'))
GENERAL_IMAGES = sorted(GENERAL_DIR.glob('*.jpg'))

assert FACE_IMAGES, 'No face fixtures found in tests/fixtures/faces'
assert GENERAL_IMAGES, 'No general fixtures found in tests/fixtures/general'

print(f'Face fixtures: {len(FACE_IMAGES)}')
print(f'General fixtures: {len(GENERAL_IMAGES)}')
print('Face files:', [p.name for p in FACE_IMAGES])
print('General files:', [p.name for p in GENERAL_IMAGES])

## Attack Parameters
Use the project default settings for evaluation consistency.

In [ ]:
params = CloakHyperparameters(
    epsilon=0.03,
    alpha_fraction=0.10,
    num_steps=100,
    l2_lambda=0.01,
    face_weight=1.0,
    clip_weight=1.0,
)
clip_model = get_clip_model()
params

## Face Biometric Cloaking on All Face Fixtures
For each face fixture, run the combined FaceNet+CLIP attack and report FaceNet/CLIP residual similarity plus SSIM.

In [ ]:
face_rows = []

for path in FACE_IMAGES:
    image = Image.open(path).convert('RGB')
    route = classify_image_type(image)
    if route.image_type != 'face':
        print(f'Skipping {path.name}: not classified as face')
        continue

    detected = route.detected_face or detect_primary_face(image)
    face_orig = extract_embedding_numpy(detected.tensor)
    clip_orig = extract_clip_embedding_numpy(detected.image)

    result = cloak_face_tensor(detected.tensor, clip_model=clip_model, parameters=params)

    face_cloak = extract_embedding_numpy(result.cloaked_face_tensor)
    clip_cloak = extract_clip_embedding_numpy(result.cloaked_face_image)

    face_sim = cosine_similarity(face_orig, face_cloak)
    clip_sim = cosine_similarity(clip_orig, clip_cloak)
    ssim = compute_ssim_score(detected.image, result.cloaked_face_image)

    face_rows.append((path.name, face_sim, clip_sim, ssim, detected.image, result.cloaked_face_image))

print(f'Processed {len(face_rows)} face fixtures')

In [ ]:
for name, face_sim, clip_sim, ssim, _, _ in face_rows:
    print(f'{name:20s} | FaceNet: {face_sim: .4f} | CLIP: {clip_sim: .4f} | SSIM: {ssim: .4f}')

if face_rows:
    print('---')
    print('Face mean FaceNet residual:', float(np.mean([r[1] for r in face_rows])))
    print('Face mean CLIP residual   :', float(np.mean([r[2] for r in face_rows])))
    print('Face mean SSIM            :', float(np.mean([r[3] for r in face_rows])))

In [ ]:
if face_rows:
    fig, axes = plt.subplots(len(face_rows), 2, figsize=(8, 4 * len(face_rows)))
    if len(face_rows) == 1:
        axes = np.array([axes])
    for i, (name, face_sim, clip_sim, ssim, orig_img, cloak_img) in enumerate(face_rows):
        axes[i, 0].imshow(orig_img)
        axes[i, 0].set_title(f'{name} - Original')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(cloak_img)
        axes[i, 1].set_title(f'FaceNet {face_sim:.3f}, CLIP {clip_sim:.3f}, SSIM {ssim:.3f}')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

## Universal Cloaking on All General Fixtures
For each general fixture, run CLIP-space universal cloaking and report residual CLIP similarity plus SSIM.

In [ ]:
general_rows = []

for path in GENERAL_IMAGES:
    image = Image.open(path).convert('RGB')
    clip_orig = extract_clip_embedding_numpy(image)

    result = cloak_general_image(image, clip_model=clip_model, parameters=params)
    clip_cloak = extract_clip_embedding_numpy(result.cloaked_image)

    clip_sim = cosine_similarity(clip_orig, clip_cloak)
    ssim = compute_ssim_score(image, result.cloaked_image)

    general_rows.append((path.name, clip_sim, ssim, image, result.cloaked_image))

print(f'Processed {len(general_rows)} general fixtures')

In [ ]:
for name, clip_sim, ssim, _, _ in general_rows:
    print(f'{name:20s} | CLIP residual: {clip_sim: .4f} | SSIM: {ssim: .4f}')

if general_rows:
    print('---')
    print('General mean CLIP residual:', float(np.mean([r[1] for r in general_rows])))
    print('General mean SSIM         :', float(np.mean([r[2] for r in general_rows])))

In [ ]:
if general_rows:
    fig, axes = plt.subplots(len(general_rows), 2, figsize=(8, 4 * len(general_rows)))
    if len(general_rows) == 1:
        axes = np.array([axes])
    for i, (name, clip_sim, ssim, orig_img, cloak_img) in enumerate(general_rows):
        axes[i, 0].imshow(orig_img)
        axes[i, 0].set_title(f'{name} - Original')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(cloak_img)
        axes[i, 1].set_title(f'CLIP {clip_sim:.3f}, SSIM {ssim:.3f}')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

## Final Validation Snapshot
This final table gives the compact cross-domain validation outcome over all fixtures.

In [ ]:
face_mean_clip = float(np.mean([r[2] for r in face_rows])) if face_rows else math.nan
face_mean_ssim = float(np.mean([r[3] for r in face_rows])) if face_rows else math.nan
general_mean_clip = float(np.mean([r[1] for r in general_rows])) if general_rows else math.nan
general_mean_ssim = float(np.mean([r[2] for r in general_rows])) if general_rows else math.nan

print('Domain            | Mean Residual Similarity | Mean SSIM')
print('------------------|---------------------------|----------')
print(f'Face (CLIP)       | {face_mean_clip: .4f}                  | {face_mean_ssim: .4f}')
print(f'General (CLIP)    | {general_mean_clip: .4f}                  | {general_mean_ssim: .4f}')